# Hands-on 29 (Bab 11): Mini Proyek Klasifikasi Sederhana di Raspberry Pi / Emulator

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 11.

Mini proyek end-to-end meniru deployment edge: simulator sensor -> model lokal
(joblib dari Hands-on 27) -> status klasifikasi -> log CSV dan/atau MQTT.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

[INFO] Working directory: /home/sandbox/iot-ml-book


## 1. Mini Proyek tanpa MQTT (simulator -> model -> log CSV)

In [2]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/edge_room_classifier_project.py", "--steps", "15", "--delay", "0"], capture_output=True, text=True)
print(r.stdout)

=== Mini proyek edge room classifier dimulai ===
step=0 | T=29.75 H=40.75 CO2=1013 P=148.98 -> status=warning
step=1 | T=25.26 H=43.07 CO2=628 P=218.1 -> status=normal
step=2 | T=24.29 H=42.81 CO2=926 P=201.07 -> status=normal
step=3 | T=24.24 H=45.97 CO2=1309 P=144.09 -> status=warning
step=4 | T=29.3 H=64.28 CO2=463 P=251.76 -> status=warning
step=5 | T=25.44 H=52.68 CO2=1019 P=131.1 -> status=warning
step=6 | T=32.61 H=50.1 CO2=639 P=175.99 -> status=warning
step=7 | T=27.23 H=50.32 CO2=991 P=261.43 -> status=normal
step=8 | T=30.57 H=56.09 CO2=1225 P=115.76 -> status=warning
step=9 | T=26.64 H=58.86 CO2=1190 P=215.47 -> status=normal
step=10 | T=30.34 H=41.37 CO2=916 P=254.61 -> status=warning
step=11 | T=32.87 H=65.66 CO2=656 P=176.03 -> status=warning
step=12 | T=28.08 H=65.02 CO2=783 P=174.04 -> status=normal
step=13 | T=25.89 H=48.01 CO2=596 P=221.83 -> status=normal
step=14 | T=25.54 H=61.87 CO2=784 P=192.45 -> status=normal
[SUKSES] Log hasil inferensi disimpan ke: outputs/re

## 2. Verifikasi Log CSV

In [3]:
import pandas as pd
log = pd.read_csv("outputs/reports/edge_room_predictions.csv")
print(log.head(10).to_string(index=False))
print("\nDistribusi status:", log["predicted_status"].value_counts().to_dict())

 step  temperature_c  humidity_pct  co2_ppm  power_watt predicted_status
    0          29.75         40.75     1013      148.98          warning
    1          25.26         43.07      628      218.10           normal
    2          24.29         42.81      926      201.07           normal
    3          24.24         45.97     1309      144.09          warning
    4          29.30         64.28      463      251.76          warning
    5          25.44         52.68     1019      131.10          warning
    6          32.61         50.10      639      175.99          warning
    7          27.23         50.32      991      261.43           normal
    8          30.57         56.09     1225      115.76          warning
    9          26.64         58.86     1190      215.47           normal

Distribusi status: {'warning': 8, 'normal': 7}


## 3. Versi dengan MQTT (broker lokal)

In [4]:
import time

broker = subprocess.Popen([sys.executable, "scripts/mqtt/run_broker.py"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
try:
    sub = subprocess.Popen(
        [sys.executable, "scripts/mqtt/subscribe_sensor.py",
         "--topic", "iotlab/room01/prediction", "--max-messages", "6", "--timeout", "25"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    time.sleep(1)
    pub = subprocess.run([sys.executable, "scripts/edge_room_classifier_mqtt.py",
                          "--steps", "6", "--delay", "0"], capture_output=True, text=True)
    print(pub.stdout)
    out, _ = sub.communicate()
    print("--- Output subscriber ---")
    print(out[:800])
finally:
    broker.terminate()

=== Mini proyek edge room classifier + MQTT dimulai ===
step=0 | publish iotlab/room01/prediction -> status=warning
step=1 | publish iotlab/room01/prediction -> status=normal
step=2 | publish iotlab/room01/prediction -> status=normal
step=3 | publish iotlab/room01/prediction -> status=warning
step=4 | publish iotlab/room01/prediction -> status=warning
step=5 | publish iotlab/room01/prediction -> status=warning
[SUKSES] Semua prediksi dipublish ke broker.

--- Output subscriber ---
[INFO] Terhubung ke broker MQTT.
[INFO] Subscribe ke topik: iotlab/room01/prediction

=== Pesan baru diterima ===
Topik   : iotlab/room01/prediction
QoS     : 1
Payload : {"step": 0, "device_id": "room01", "temperature_c": 29.75, "humidity_pct": 40.75, "co2_ppm": 1013, "power_watt": 148.98, "predicted_status": "warning"}
[SUKSES] Payload JSON valid.
Device ID     : room01
Temperature C : 29.75
Humidity %    : 40.75
CO2 ppm       : 1013
Power watt    : 148.98
Timestamp     : None

=== Pesan baru diterima ===
T

## Penutup

Mini proyek menyatukan seluruh konsep Bab 11: training di laptop, ekspor model,
inferensi lokal per sampel, lalu hasil diteruskan ke log CSV dan/atau broker MQTT.
Prediksi yang diterbitkan ke topik `iotlab/room01/prediction` terverifikasi diterima
subscriber. Log simulasi tersimpan di `outputs/reports/edge_room_predictions.csv`.